# Live experiment

Copy this folder, rename it, and work through the cells marked **TODO**. As shipped, the notebook runs end to end on the virtual microscope, so you can check that your environment works before you change anything. Each TODO cell holds a working default and lists the real-scope alternatives as comments.

The [live experiment example](https://github.com/pertzlab/faro/blob/main/examples/live_experiment.ipynb) runs the same structure with explanations for every step. The [README](https://github.com/pertzlab/faro/blob/main/README.md) is the reference.

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import faro.core.utils as utils
from faro.core.controller import Controller
from faro.core.data_structures import RTMSequence, SegmentationMethod, combine
from faro.core.pipeline import ImageProcessingPipeline
from faro.core.utils import events_to_dataframe
from faro.core.writers import OmeZarrWriter

## TODO: microscope

Replace the virtual microscope with yours. Pertzlab scopes live in `faro.microscope.pertzlab`; for a new scope see [Adding Your Own Micro-Manager Microscope](https://github.com/pertzlab/faro/blob/main/README.md#adding-your-own-micro-manager-microscope). Nothing else in the notebook depends on which one you pick.

In [ ]:
from virtual_microscope.backends.optogenetic import setup_optogenetic
from faro.microscope.simulation import UniMMCoreSimulation

core, sim = setup_optogenetic()
mic = UniMMCoreSimulation(mmc=core)
mic.init_scope()

# from faro.microscope.pertzlab.moench import Moench;   mic = Moench(None)
# from faro.microscope.pertzlab.niesen import Niesen;   mic = Niesen(None)
# from faro.microscope.demo import MMDemo;              mic = MMDemo()

## Open napari

Live view, stage control and the position list come from napari-micromanager. The status widget shows the run once a controller is bound to it below.

In [ ]:
import napari
from napari_micromanager import MainWindow
from faro.widgets import ExperimentStatusWidget

viewer = napari.Viewer()
mm_widget = MainWindow(viewer, mmcore=mic.mmc)
viewer.window.add_dock_widget(mm_widget, name="napari-micromanager")

status_widget = ExperimentStatusWidget()
viewer.window.add_dock_widget(status_widget, name="experiment status", area="right")

## TODO: experiment settings

Where results go, how long the phases are, and which channels to use. Channel `config` names must exist in your Micro-Manager channel group; use `PowerChannel` when the stimulation channel has a power setting.

In [ ]:
EXPERIMENT_NAME = "2026-01-01_my_experiment"
STORAGE_ROOT = tempfile.mkdtemp()            # TODO: a real folder, e.g. r"D:\data"

INTERVAL_S = 1.0                             # TODO: seconds between frames, e.g. 30
N_BASELINE = 5                               # frames before stimulation
N_STIM = 5                                   # frames with stimulation
N_RECOVERY = 5                               # frames after stimulation

IMAGING_CHANNELS = [{"config": "phase-contrast", "exposure": 50}]   # TODO: e.g. {"config": "miRFP", "exposure": 300}
STIM_CHANNEL = {"config": "phase-contrast", "exposure": 50}         # TODO: e.g. {"config": "CyanStim", "exposure": 200}
# from faro.core.data_structures import PowerChannel
# STIM_CHANNEL = PowerChannel(config="CyanStim", exposure=200, power=10)

## TODO: set up the pipeline

Segmentation, tracking, feature extraction and stimulation. Every `SegmentationMethod` needs a name; the feature extractor and stimulator refer to it. Cellpose needs `uv sync --extra cellpose`. Set `stimulator = None` for an experiment without feedback. A stimulator may declare `required_metadata`; those keys go into the `rtm_metadata` of the stimulation phase below. For your own components see [Writing your own components](https://github.com/pertzlab/faro/blob/main/README.md#writing-your-own-components).

In [ ]:
from faro.segmentation.base import OtsuSegmentator
from faro.feature_extraction.simple import SimpleFE
from faro.stimulation.percentage_of_cell import StimPercentageOfCell
from faro.tracking.trackpy import TrackerTrackpy

# Segmentation. Alternatives:
# from faro.segmentation.cellpose_v4 import CellposeV4;   seg = CellposeV4(diameter=30)
seg = OtsuSegmentator()
segmentators = [
    SegmentationMethod(name="labels", segmentation_class=seg, use_channel=0, save_tracked=True),
]

# Features. Alternatives:
# from faro.feature_extraction.erk_ktr import FE_ErkKtr;  feature_extractor = FE_ErkKtr("labels")
feature_extractor = SimpleFE("labels")

# Stimulation. Alternatives:
# from faro.stimulation.base import StimWholeFOV;          stimulator = StimWholeFOV()
# stimulator = None                                        # no feedback, plain timelapse
stimulator = StimPercentageOfCell()

# Tracking: search_range is the largest centroid movement between frames, in pixels.
tracker = TrackerTrackpy(search_range=30)

path = os.path.join(STORAGE_ROOT, EXPERIMENT_NAME)
os.makedirs(path, exist_ok=True)
pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
)
ctrl = Controller(mic, pipeline, writer=OmeZarrWriter(storage_path=path))
print("results go to", path)

In [ ]:
status_widget.set_controller(ctrl)

## DMD calibration

Scopes with a DMD need a fresh calibration every session so stimulation masks land on the right pixels. `calibrate_dmd` always runs when called and does nothing on a scope without a DMD. The calibration light hits your sample, so move to an empty area first.

In [ ]:
if stimulator is not None:
    mic.calibrate_dmd(STIM_CHANNEL["config"])

## Preview

Snap one frame at the current position and check labels and stimulation mask before committing to a long run.

In [ ]:
channel = IMAGING_CHANNELS[0]["config"]
mic.mmc.setConfig(mic.resolve_group(channel), channel)
mic.mmc.snapImage()
test_img = mic.mmc.getImage()
labels_preview = segmentators[0].segmentation_class.segment(test_img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_img, cmap="gray")
axes[0].set_title("Raw image")
axes[1].imshow(labels_preview, cmap="nipy_spectral")
axes[1].set_title(f"Labels ({labels_preview.max()} cells)")
axes[2].imshow(test_img, cmap="gray")
if stimulator is not None:
    from faro.core.pipeline import dispatch_stim_mask
    mask_preview = dispatch_stim_mask(
        stimulator, {"labels": labels_preview},
        {"img_shape": test_img.shape, "stim_cell_percentage": 0.3}, img=test_img[None]
    )
    if mask_preview is True:                       # StimWholeFOV
        mask_preview = np.ones_like(labels_preview)
    axes[2].imshow(np.ma.masked_where(mask_preview == 0, mask_preview), cmap="autumn", alpha=0.8)
axes[2].set_title("Stimulation mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## TODO: positions

Pick positions in the napari-micromanager MDA widget and read them with `generate_fov_positions(mic, viewer=viewer)`, or list them by hand.

In [ ]:
fov_positions = utils.generate_fov_positions_from_list(mic, [{"x": 0.0, "y": 0.0, "z": 0.0}])
# fov_positions = utils.generate_fov_positions(mic, viewer=viewer)   # from the MDA widget

## Build the event list

Baseline, stimulation and recovery phases joined in time. Delete or add phases as needed; `rtm_metadata` ends up in the tracks so you can group by phase.

In [ ]:
baseline = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_BASELINE},
    stage_positions=fov_positions,
    channels=IMAGING_CHANNELS,
    rtm_metadata={"phase": "baseline"},
)
stim_phase = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_STIM},
    stage_positions=fov_positions,
    channels=IMAGING_CHANNELS,
    stim_channels=[STIM_CHANNEL] if stimulator is not None else None,
    stim_frames=range(N_STIM) if stimulator is not None else None,
    # metadata travels with every event and reaches the stimulator; StimPercentageOfCell
    # reads stim_cell_percentage from it (validate_events warns when it is missing)
    rtm_metadata={"phase": "stimulation", "stim_cell_percentage": 0.3},
)
recovery = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_RECOVERY},
    stage_positions=fov_positions,
    channels=IMAGING_CHANNELS,
    rtm_metadata={"phase": "recovery"},
)
events = combine(baseline, stim_phase, recovery, axis="t")
df_events = events_to_dataframe(events)
print(f"{len(events)} events over {df_events['time'].max() / 60:.1f} min")
df_events.head()

## Validate and load

Fix every warning before you start. Validation checks the pipeline components, the event metadata, channel names, exposure limits and the DMD calibration. `load_experiment` then shows the plan in the status widget.

In [ ]:
assert ctrl.validate_events(events), "fix the warnings above before running"
ctrl.load_experiment(events, stim_mode="current")

## Run

Start from the widget or with the next cell. The kernel stays free during the run.

In [ ]:
handle = ctrl.start_experiment()

In [ ]:
handle.status()

## Finish

Blocks until the last frame is processed, then writes `exp_data.parquet` with all fields of view combined.

In [ ]:
final = handle.wait()
ctrl.finish_experiment()
mic.post_experiment()
utils.generate_exp_data_from_tracks(path)
print(f"state {final.state!r}, {final.n_frames_received} frames, "
      f"{len(final.background_errors)} background errors")
pd.read_parquet(os.path.join(path, "exp_data.parquet")).head()